In [1]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('../datasets/kidney.csv')

# Display the first few rows
display(df.head())

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [2]:
# 3. Replace garbage strings with official pandas missing values (NaN)
df = df.replace('?', np.nan)
df = df.replace('\t?', np.nan)

# 4. Count how many missing values are in each column
print("Total missing values per column:")
print(df.isnull().sum())

Total missing values per column:
id                  0
age                 9
bp                 12
sg                 47
al                 46
su                 49
rbc               152
pc                 65
pcc                 4
ba                  4
bgr                44
bu                 19
sc                 17
sod                87
pot                88
hemo               52
pcv                71
wc                106
rc                131
htn                 2
dm                  2
cad                 2
appet               1
pe                  1
ane                 1
classification      0
dtype: int64


In [5]:
from pandas.api.types import is_numeric_dtype

# 1. Safely convert columns to numbers where possible
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except Exception:
        pass

# 2. Fill the missing values intelligently
for col in df.columns:
    if is_numeric_dtype(df[col]):
        # If it is a number column, fill with the average (mean)
        df[col] = df[col].fillna(df[col].mean())
    else:
        # If it is a text column, fill with the most frequent word (mode)
        df[col] = df[col].fillna(df[col].mode()[0])

# 3. Convert all remaining text into numbers for Machine Learning (Label Encoding)
for col in df.columns:
    if not is_numeric_dtype(df[col]):
        df[col] = df[col].astype('category').cat.codes

# 4. Verify that all blanks are filled
print("Total missing values left:", df.isnull().sum().sum())

Total missing values left: 0


In [8]:
import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle

df = pd.read_csv('../datasets/kidney.csv')

# 🚨 THE FIX: Drop the ID column so the AI can't cheat!
df = df.drop(['id'], axis=1, errors='ignore')

for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except Exception:
        pass

for col in df.columns:
    if is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].mean())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

for col in df.columns:
    if not is_numeric_dtype(df[col]):
        df[col] = df[col].astype('category').cat.codes

target_col = 'classification' if 'classification' in df.columns else 'class'
X = df.drop(target_col, axis=1)
y = df[target_col]

temp_model = RandomForestClassifier(random_state=42)
temp_model.fit(X, y)

importances = pd.Series(temp_model.feature_importances_, index=X.columns)
top_6_ckd_features = importances.nlargest(6).index.tolist()
print("Top 6 CKD Questions:", top_6_ckd_features)

X_optimized = X[top_6_ckd_features]
X_train, X_test, y_train, y_test = train_test_split(X_optimized, y, test_size=0.2, random_state=42)

final_ckd_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_ckd_model.fit(X_train, y_train)
print(f"True Optimized CKD Accuracy: {final_ckd_model.score(X_test, y_test) * 100:.2f}%")

with open('../models/ckd_model.pkl', 'wb') as file:
    pickle.dump(final_ckd_model, file)

Top 6 CKD Questions: ['sg', 'hemo', 'sc', 'pcv', 'al', 'bgr']
True Optimized CKD Accuracy: 98.75%
